# Transcribe & Pre-fill Firebase Captions

Processes **one video** at a time:
1. Downloads audio from YouTube
2. Transcribes with faster-whisper (large-v3, Ukrainian)
3. Re-segments into sentence-level captions (splits multi-sentence, merges fragments)
4. Outputs Firebase-ready JSON for `videos/` and `video_captions/`

In [ ]:
!pip install -q faster-whisper yt-dlp[default]
!pip install -q firebase-admin
# yt-dlp needs a JS runtime for YouTube extraction — install deno (its default runtime)
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh > /dev/null 2>&1
!deno --version

In [ ]:
import re
import json
import subprocess
from pathlib import Path
from faster_whisper import WhisperModel

## 1. Configuration

In [ ]:
VIDEO_URL = "https://youtu.be/gtFOLKjnays"

MODEL_SIZE = "large-v3"
LANGUAGE = "uk"
FIREBASE_DB_URL = "https://usleditordatabase-default-rtdb.europe-west1.firebasedatabase.app/"

AUDIO_DIR = Path("audio")
OUTPUT_DIR = Path("firebase_prefill")
AUDIO_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# Extract video ID
match = re.search(r'(?:v=|youtu\.be/)([a-zA-Z0-9_-]{11})', VIDEO_URL)
VIDEO_ID = match.group(1) if match else VIDEO_URL
print(f"Video ID: {VIDEO_ID}")

## 2. Download Audio

In [ ]:
audio_path = AUDIO_DIR / f"{VIDEO_ID}.wav"

if audio_path.exists():
    print(f"Already downloaded: {audio_path}")
else:
    !yt-dlp -x --audio-format wav -o "{AUDIO_DIR}/{VIDEO_ID}.%(ext)s" --no-playlist "{VIDEO_URL}"
    if not audio_path.exists():
        raise FileNotFoundError(
            f"Download failed — {audio_path} not found.\n"
            "Check the yt-dlp output above for errors.\n"
            "Common fixes: update yt-dlp (!pip install -U yt-dlp[default]), "
            "or check if the video URL is valid/accessible."
        )
    print(f"Saved: {audio_path}")

## 3. Transcribe

In [ ]:
model = WhisperModel(MODEL_SIZE, device="auto", compute_type="auto")
print(f"Model loaded: {MODEL_SIZE}")

In [ ]:
segments_iter, info = model.transcribe(
    str(audio_path),
    language=LANGUAGE,
    vad_filter=True,
)
print(f"Language: {info.language} (prob: {info.language_probability:.2f})")

raw_segments = []
for seg in segments_iter:
    raw_segments.append({
        "start": round(seg.start, 2),
        "end": round(seg.end, 2),
        "text": seg.text.strip(),
    })

print(f"{len(raw_segments)} raw whisper segments")
for s in raw_segments[:10]:
    print(f"  [{s['start']:>7.2f} - {s['end']:>7.2f}] {s['text']}")
if len(raw_segments) > 10:
    print(f"  ... and {len(raw_segments) - 10} more")

## 4. Re-segment into Sentences

Whisper segments don't respect sentence boundaries:
- One segment may contain multiple sentences
- One sentence may be split across segments

This step splits on sentence-ending punctuation (`.` `!` `?`) and
merges fragments that don't end with punctuation into the next segment.

In [ ]:
def resegment_sentences(raw_segments):
    """Split multi-sentence segments and merge fragments into full sentences.
    
    Uses word-level timestamp interpolation: within a whisper segment,
    timestamps are distributed proportionally by character count.
    """
    if not raw_segments:
        return []

    # Step 1: split each segment into sub-sentences with interpolated timestamps
    pieces = []  # list of (start, end, text)
    for seg in raw_segments:
        text = seg["text"].strip()
        if not text:
            continue
        # Split on sentence-ending punctuation, keeping the punctuation
        parts = re.split(r'(?<=[.!?])\s+', text)
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) <= 1:
            pieces.append((seg["start"], seg["end"], text))
        else:
            # Interpolate timestamps proportionally by character length
            total_chars = sum(len(p) for p in parts)
            t = seg["start"]
            duration = seg["end"] - seg["start"]
            for p in parts:
                frac = len(p) / total_chars
                p_end = t + duration * frac
                pieces.append((round(t, 2), round(p_end, 2), p))
                t = p_end

    # Step 2: merge fragments (pieces not ending with sentence punctuation)
    # into the next piece to form complete sentences
    sentences = []
    buf_start = None
    buf_text = ""

    for start, end, text in pieces:
        if buf_start is None:
            buf_start = start
        buf_text = (buf_text + " " + text).strip()

        # Check if this piece ends a sentence
        if re.search(r'[.!?]$', buf_text):
            sentences.append({
                "start_time": buf_start,
                "end_time": end,
                "text": buf_text,
                "aligned": False,
            })
            buf_start = None
            buf_text = ""

    # Flush remaining buffer (no trailing punctuation)
    if buf_text:
        sentences.append({
            "start_time": buf_start,
            "end_time": pieces[-1][1],
            "text": buf_text,
            "aligned": False,
        })

    return sentences


sentences = resegment_sentences(raw_segments)
print(f"{len(raw_segments)} whisper segments -> {len(sentences)} sentences\n")
for s in sentences[:15]:
    print(f"  [{s['start_time']:>7.2f} - {s['end_time']:>7.2f}] {s['text']}")
if len(sentences) > 15:
    print(f"  ... and {len(sentences) - 15} more")

## 5. Build Firebase JSON

In [ ]:
# videos/{videoId}
videos_entry = {
    VIDEO_ID: {
        "url": VIDEO_URL,
        "complete": False,
    }
}

# video_captions/{videoId}
captions_entry = {
    VIDEO_ID: {
        "captions": sentences,
    }
}

# Save to fixed filenames (overwritten each run)
videos_path = OUTPUT_DIR / "videos.json"
captions_path = OUTPUT_DIR / "captions.json"

with open(videos_path, "w", encoding="utf-8") as f:
    json.dump(videos_entry, f, ensure_ascii=False, indent=2)

with open(captions_path, "w", encoding="utf-8") as f:
    json.dump(captions_entry, f, ensure_ascii=False, indent=2)

print(f"Saved: {videos_path}")
print(f"Saved: {captions_path}")
print(f"\n{len(sentences)} captions for {VIDEO_ID}")

## 6. Preview Firebase Output

In [ ]:
print("=== videos entry ===")
print(json.dumps(videos_entry, ensure_ascii=False, indent=2))

print(f"\n=== video_captions entry ({len(sentences)} captions) ===")
# Show first 5 captions
preview = {VIDEO_ID: {**captions_entry[VIDEO_ID], "captions": sentences[:5]}}
print(json.dumps(preview, ensure_ascii=False, indent=2))
if len(sentences) > 5:
    print(f"  ... and {len(sentences) - 5} more captions")

## 7. Upload to Firebase

Needs the service account key (`serviceAccountKey.json`) uploaded to Colab.

In [ ]:
def upload_to_firebase(service_account_key="serviceAccountKey.json"):
    """Upload videos and captions JSON to Firebase Realtime Database.
    
    Assigns integer IDs to videos/ by reading the current max index from the DB.
    video_captions/ are keyed by YouTube video ID.
    Skips if a video with the same URL already exists.
    """
    import firebase_admin
    from firebase_admin import credentials, db

    if not firebase_admin._apps:
        cred = credentials.Certificate(service_account_key)
        firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_DB_URL})

    with open(str(videos_path), "r", encoding="utf-8") as f:
        videos = json.load(f)

    with open(str(captions_path), "r", encoding="utf-8") as f:
        captions = json.load(f)

    # Firebase returns a list when all keys are sequential integers
    existing_videos = db.reference("videos").get() or {}
    if isinstance(existing_videos, list):
        existing_videos = {str(i): v for i, v in enumerate(existing_videos) if v is not None}

    existing_urls = {v["url"] for v in existing_videos.values() if isinstance(v, dict) and "url" in v}

    # Find next integer ID
    existing_ids = [int(k) for k in existing_videos.keys() if str(k).isdigit()]
    next_id = max(existing_ids, default=-1) + 1

    for yt_id in videos:
        url = videos[yt_id]["url"]
        if url in existing_urls:
            print(f"SKIP {yt_id} — URL already exists in DB")
            continue

        db.reference(f"videos/{next_id}").set(videos[yt_id])
        db.reference(f"video_captions/{yt_id}").set(captions[yt_id])
        print(f"Uploaded video {next_id} ({yt_id}) + captions.")
        next_id += 1

In [ ]:
upload_to_firebase()